In [ ]:
import os
from datetime import datetime, timezone

import pandas as pd
import requests
from IPython.display import display

API_KEY = os.environ.get("PURPLEAIR_API_KEY")
if not API_KEY:
    raise SystemExit(
        "Set PURPLEAIR_API_KEY to a PurpleAir READ API key, then rerun this cell."
    )

# Approximate New Orleans metro bounding box.
NW_LAT = 30.10
NW_LNG = -90.25
SE_LAT = 29.86
SE_LNG = -89.85

url = "https://api.purpleair.com/v1/sensors"
headers = {"X-API-Key": API_KEY}
params = {
    "fields": "sensor_index,name,pm2.5,last_seen,latitude,longitude,confidence",
    "nwlat": NW_LAT,
    "nwlng": NW_LNG,
    "selat": SE_LAT,
    "selng": SE_LNG,
}

response = requests.get(url, headers=headers, params=params, timeout=30)
response.raise_for_status()
payload = response.json()

if "error" in payload:
    raise RuntimeError(f"PurpleAir API error: {payload['error']} - {payload.get('description', '')}")

fields = payload.get("fields")
rows = payload.get("data", [])

if not rows:
    print("No sensors were returned for the selected New Orleans bounding box.")
else:
    if isinstance(rows[0], dict):
        frame = pd.DataFrame(rows)
    elif fields:
        frame = pd.DataFrame(rows, columns=fields)
    else:
        frame = pd.DataFrame(rows)

    if "last_seen" in frame.columns:
        frame["last_seen_utc"] = pd.to_datetime(frame["last_seen"], unit="s", utc=True)

    preferred_columns = [
        column
        for column in [
            "sensor_index",
            "name",
            "pm2.5",
            "confidence",
            "latitude",
            "longitude",
            "last_seen_utc",
        ]
        if column in frame.columns
    ]

    print(f"Returned {len(frame)} sensor(s) from the New Orleans bounding box.")
    display(frame[preferred_columns].sort_values(by=["pm2.5"], ascending=False) if "pm2.5" in frame.columns else frame[preferred_columns])

    latest_seen = frame["last_seen_utc"].max() if "last_seen_utc" in frame.columns else None
    if latest_seen is not None:
        print(f"Most recent reading seen at: {latest_seen.to_pydatetime().astimezone(timezone.utc).isoformat()}")


Returned 17 sensor(s) from the New Orleans bounding box.


,sensor_index,name,pm2.5,confidence,latitude,longitude,last_seen_utc
5,287258,Operation Spark PurpleAir - New Orleans East,7.6,100,30.031713,-89.954830,2026-09-14 16:56:06+00:00
8,307890,JosephAQI,7.0,100,29.923140,-90.116810,2026-09-14 16:56:58+00:00
2,278265,Magazine by Whole Foods,6.7,100,29.921090,-90.116490,2026-09-14 16:56:36+00:00
3,16289,St. Bernard,6.1,100,29.954208,-89.988740,2026-09-14 16:56:05+00:00
1,5424,River Ridge,5.7,100,29.965136,-90.219770,2026-09-14 16:56:13+00:00
13,198085,English Turn,5.1,100,29.896032,-89.938750,2026-09-14 16:57:04+00:00
12,192451,Uptown PurpleAir Zen Sensor,5.1,100,29.919237,-90.104290,2026-09-14 16:57:02+00:00
9,98161,Irish Channel,5.0,100,29.920322,-90.083490,2026-09-14 16:57:58+00:00
7,292843,New Orleans Warehouse District,4.8,100,29.945938,-90.067750,2026-09-14 16:54:08+00:00
16,260283,Xavier Univeristy_Admin Building_V2,4.6,100,29.964190,-90.107346,2026-09-14 16:56:35+00:00


Most recent reading seen at: 2026-09-14T16:57:58+00:00
